In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import missingno as msno

In [ ]:
DATA_PATH = "/Volumes/DATASHURPRO/Bordeaux_CHU/"

In [ ]:
### Paramètres vitaux ###

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re

# --- Étape 1 : Concaténation de tous les fichiers ---
DATA_PATH = "Bordeaux_CHU/"
pattern = os.path.join(DATA_PATH, "CGJ 052c -*")
files = glob.glob(pattern)

if not files:
    print("Aucun fichier correspondant trouvé.")
    exit()

print("Fichiers trouvés :")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
for file in files:
    try:
        # On saute les 3 premières lignes pour obtenir l'en-tête désiré
        df = pd.read_excel(file, skiprows=3)
        dfs.append(df)
    except Exception as e:
        print(f"Erreur lors de la lecture de {os.path.basename(file)} : {e}")

if not dfs:
    print("Aucun fichier n'a pu être traité.")
    exit()

df_concat = pd.concat(dfs, ignore_index=True)
print(f"\nConcaténation terminée. Nombre total de lignes : {len(df_concat)}")


# --- Étape 2 : Fusion des colonnes via regex ---
def merge_columns_regex(df, regex_candidates, new_col_name):
    """
    Pour chaque ligne, recherche parmi les colonnes du DataFrame celles dont le nom
    correspond à l'un des motifs regex dans regex_candidates (insensible à la casse)
    et fusionne (combine_first) les valeurs non manquantes dans une nouvelle colonne.
    """
    series = None
    for pattern in regex_candidates:
        matching_cols = [col for col in df.columns if re.search(pattern, col, re.IGNORECASE)]
        for col in matching_cols:
            if series is None:
                series = df[col]
            else:
                series = series.combine_first(df[col])
    df[new_col_name] = series
    return df

# Copie pour la fusion
df_merged = df_concat.copy()

# Dictionnaire de correspondance : chaque variable cible est associée à une liste de motifs regex
columns_mapping = {
    "nda": [r"^Unnamed\s*:\s*2$"],  # nda provenant uniquement de Unnamed: 2
    "date_heure_adm": [r"^Unnamed\s*:\s*3$"],  # date d'admission provenant uniquement de Unnamed: 3
    "date_param": [r"Date\s*et\s*Heure"],  # date_param (différent de date_heure_adm)
    "Bandelette Urinaire": [r"^Bandelette\s+Urinaire$"],  # conserver la colonne telle quelle
    "tas": [r"TA\s*[-\s]*Max"],
    "tad": [r"TA\s*[-\s]*Min"],
    "fc": [r"Fr[ée]quence\s+Cardiaque"],
    "temp": [r"Temp[ée]rature\s*\(°C\)"],
    "sat": [r"Sp02\s*concat", r"Saturation\s*(P[ée]riph[ée]rique\s*en\s*O2|En\s*O2)"],
    "fr": [r"Fr[ée]quence\s+Respiratoire"],
    "glycemie": [r"Glyc[ée]mie\s+(Digitale\s*\(gr/l\)|digital)"],
    "hemocue": [r"Hemocue"],
    "gcs": [r"Glasgow", r"Score\s+de\s+Glasgow"],
    "eval_douleur": [r"EN\.", r"Echelle\s+de\s+douleur"],
    "OH_expi": [r"Ethylotest"]
}

# Appliquer la fusion pour chaque variable
for new_col, regex_list in columns_mapping.items():
    df_merged = merge_columns_regex(df_merged, regex_list, new_col)

# --- Sélection des colonnes finales ---
final_cols = [
    "nda", "date_heure_adm", "date_param", "Bandelette Urinaire",
    "tas", "tad", "fc", "temp", "sat", "fr", "glycemie", "hemocue",
    "gcs", "eval_douleur", "OH_expi"
]
df_final = df_merged[[col for col in final_cols if col in df_merged.columns]].copy()

# --- Conversion des dates ---
df_final["date_param"] = pd.to_datetime(df_final["date_param"], dayfirst=True, errors="coerce")
df_final["date_heure_adm"] = pd.to_datetime(df_final["date_heure_adm"], dayfirst=True, errors="coerce")


# --- Étape 3 : Traitement des variables quantitatives ---
quant_columns = ["tas", "tad", "fc", "temp", "sat", "fr", "glycemie", "hemocue", "gcs", "eval_douleur", "OH_expi"]

def clean_numeric(x):
    """
    Nettoie une chaîne représentant un nombre :
      - Supprime les espaces superflus.
      - Si une virgule est présente, on suppose qu'elle sert de séparateur décimal.
        Dans ce cas, on supprime d'abord les points (séparateurs de milliers) et on remplace la virgule par un point.
      - Sinon, on supprime les espaces.
      - Si la chaîne est vide ou équivaut à "na", on retourne NaN.
    """
    try:
        s = str(x).strip()
        if not s or s.lower() in ["na", "n/a", ""]:
            return np.nan
        if ',' in s:
            s = s.replace('.', '')
            s = s.replace(',', '.')
        else:
            s = s.replace(' ', '')
        return s
    except Exception as e:
        return np.nan

for col in quant_columns:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(clean_numeric)
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce")
        df_final[col] = df_final[col].round(1)  # Conversion en float arrondi
        # Si vous souhaitez convertir en Int64, utilisez .astype("Int64")
        # df_final[col] = df_final[col].round().astype("Int64")

# --- Tri des données par date_heure_adm ---
df_final = df_final.sort_values(by="date_heure_adm")
df_final = df_final.dropna(how='all')
df_param_ioa = df_final.reset_index(drop=True)
df_param_ioa["nda"] = df_param_ioa["nda"].astype(str).str.replace(r'\.0$', '', regex=True)
df_param_ioa.to_csv("df_param_ioa.csv", sep = "b0L0s")

print(f"\nNombre de lignes : {len(df_final)}")

In [ ]:
#### Dossier IOA ####

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re

# --- Étape 1 : Lecture et concaténation des fichiers CGJ 030a ---
DATA_PATH = "/Bordeaux_CHU/"
pattern = os.path.join(DATA_PATH, "CGJ 030a -*")
files = glob.glob(pattern)

if not files:
    print("Aucun fichier correspondant trouvé.")
    exit()

print("Fichiers trouvés :")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
for file in files:
    try:
        # On saute les 3 premières lignes pour obtenir l'en-tête souhaité
        df = pd.read_excel(file, skiprows=3)
        dfs.append(df)
    except Exception as e:
        print(f"Erreur lors de la lecture de {os.path.basename(file)} : {e}")

if not dfs:
    print("Aucun fichier n'a pu être traité.")
    exit()

df_concat = pd.concat(dfs, ignore_index=True)
print(f"\nConcaténation terminée. Nombre total de lignes : {len(df_concat)}")

# --- Étape 2 : Renommage des colonnes positionnelles (Unnamed) ---
# Décalage de -1 selon vos recommandations :
rename_map = {
    "Unnamed: 2": "date_adm",        # colonne 2 → date_adm
    "Unnamed: 3": "nda",             # colonne 3 → nda
    "Unnamed: 4": "nom",             # colonne 4 → nom
    "Unnamed: 5": "prenom",          # colonne 5 → prenom
    "Unnamed: 6": "sexe_ioa",        # colonne 6 → sexe_ioa
    "Unnamed: 7": "age_ioa",         # colonne 7 → age_ioa
    "Unnamed: 8": "id_ioa",          # colonne 8 → id_ioa
    "Unnamed: 9": "date_tri_ioa"     # colonne 9 → date_tri_ioa_end

}
df_concat.rename(columns=rename_map, inplace=True)

# --- Renommage des autres colonnes d'après votre code R ---
other_rename = {
    "anam_ioa": "anam_ioa_1",  # colonne vide, si présente
    "Circonstances": "anam_ioa_2",
    "Commentaires aux urgences": "anam_ioa_3",
    "Histoire de la maladie": "anam_ioa_4",
    "Motif de recours": "motif_1",
    "Motif d'hospitalisation": "motif_2",
    "Motifs de recours": "motif_3",
    "Tri IAO ou Score de Gravité": "tri_ioa",
    "Antécédents": "atcd_ioa",
    "Traitement": "ttt_dos_ioa_1",
    "Traitement en cours": "ttt_dos_ioa_2",
    "Traitement administré par IOA": "ttt_adm_ioa",
}
df_concat.rename(columns=other_rename, inplace=True)

# --- Étape 3 : Création des variables calculées ---

# mode_transport : si "Mode d'arrivée aux urgences" est manquant, on prend "Mode de Transport"
if "Mode d'arrivée aux urgences" in df_concat.columns and "Mode de Transport" in df_concat.columns:
    df_concat["mode_transport"] = df_concat["Mode d'arrivée aux urgences"].combine_first(df_concat["Mode de Transport"])
elif "Mode d'arrivée aux urgences" in df_concat.columns:
    df_concat["mode_transport"] = df_concat["Mode d'arrivée aux urgences"]
elif "Mode de Transport" in df_concat.columns:
    df_concat["mode_transport"] = df_concat["Mode de Transport"]
else:
    df_concat["mode_transport"] = np.nan

# anam_ioa : priorité = anam_ioa_3, puis anam_ioa_1, puis anam_ioa_2, puis "Histoire de la maladie"
def combine_anam(row):
    if pd.notna(row.get("anam_ioa_3", np.nan)):
        return row["anam_ioa_3"]
    elif pd.notna(row.get("anam_ioa_1", np.nan)):
        return row["anam_ioa_1"]
    elif pd.notna(row.get("anam_ioa_2", np.nan)):
        return row["anam_ioa_2"]
    elif pd.notna(row.get("anam_ioa_4", np.nan)):
        return row["anam_ioa_4"]
    else:
        return np.nan

df_concat["anam_ioa"] = df_concat.apply(combine_anam, axis=1)

# motif : priorité = motif_2, puis motif_3, puis motif_1
def combine_motif(row):
    if pd.notna(row.get("motif_2", np.nan)):
        return row["motif_2"]
    elif pd.notna(row.get("motif_3", np.nan)):
        return row["motif_3"]
    elif pd.notna(row.get("motif_1", np.nan)):
        return row["motif_1"]
    else:
        return np.nan

df_concat["motif"] = df_concat.apply(combine_motif, axis=1)
# Supprimer le préfixe dans motif (tout ce qui précède et incluant "- ")
df_concat["motif"] = df_concat["motif"].astype(str).str.replace(r".*- ", "", regex=True)

# tri : calculé à partir de tri_ioa selon vos règles
def compute_tri(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["Réanimation (médecin <1min)", "Urgence absolue"]:
        return "1"
    elif s in ["Très urgent (médecin <20min)", "Urgence vraie"]:
        return "2"
    elif s in ["Urgent (médecin <1h)", "Urgence relative"]:
        return "3"
    elif s in ["Peu urgent (médecin <2h)", "Consultation urgente", "Moins urgent"]:
        return "4"
    elif s in ["Non urgent (médecin <3h)", "Consultation de médecine", "Non urgent"]:
        return "5"
    else:
        return np.nan

df_concat["tri"] = df_concat["tri_ioa"].apply(compute_tri)

# --- Étape 4 : Sélection des colonnes finales ---
final_cols = [
    "nda", "nom", "prenom", "age_ioa", "sexe_ioa", "date_adm",
    "id_ioa", "date_tri_ioa", "motif", "tri", "tri_ioa", "mode_transport", "anam_ioa",
    "anam_ioa_1", "anam_ioa_2", "anam_ioa_3", "anam_ioa_4", "atcd_ioa",
    "ttt_dos_ioa_1", "ttt_dos_ioa_2", "ttt_adm_ioa"
]
final_cols = [col for col in final_cols if col in df_concat.columns]
df_final = df_concat[final_cols].copy()

# --- Étape 5 : Conversion des dates ---
df_final["date_adm"] = pd.to_datetime(df_final["date_adm"], dayfirst=True, errors="coerce")
if "date_ioa" in df_final.columns:
    df_final["date_ioa"] = pd.to_datetime(df_final["date_ioa"], dayfirst=True, errors="coerce")
if "date_tri_ioa" in df_final.columns:
    df_final["date_tri_ioa"] = pd.to_datetime(df_final["date_tri_ioa"], dayfirst=True, errors="coerce")

# --- Étape 6 : Réinitialisation de l'index ---
df_final = df_final.sort_values(by="date_adm")
df_final = df_final.dropna(how='all')
df_ioa = df_final.reset_index(drop=True)
df_ioa["nda"] = df_ioa["nda"].astype(str).str.replace(r'\.0$', '', regex=True)
df_ioa.to_csv("df_ioa.csv")

In [ ]:
#### Dossier Med ####

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re

# --- Étape 1 : Lecture et concaténation des fichiers ---
DATA_PATH = "/Bordeaux_CHU/"
# Remplacez "NEWFILE -*" par le pattern de vos fichiers à traiter.
pattern = os.path.join(DATA_PATH, "CGJ 055b - Dossier patent full -*")
files = glob.glob(pattern)

if not files:
    print("Aucun fichier correspondant trouvé.")
    exit()

print("Fichiers trouvés :")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
for file in files:
    try:
        # On saute les 3 premières lignes pour obtenir l'en‐tête souhaité
        df = pd.read_excel(file, skiprows=1)
        dfs.append(df)
    except Exception as e:
        print(f"Erreur lors de la lecture de {os.path.basename(file)} : {e}")

if not dfs:
    print("Aucun fichier n'a pu être traité.")
    exit()

df_concat = pd.concat(dfs, ignore_index=True)
print(f"\nConcaténation terminée. Nombre total de lignes : {len(df_concat)}")


# --- Étape 2 : Renommage des colonnes "Unnamed" selon le mapping attendu ---
# D'après vos instructions, la version finale doit avoir :
# nda = Unnamed:1, nom = Unnamed:2, prenom = Unnamed:3, sexe = Unnamed:4, age = Unnamed:5,
# date_adm = Unnamed:6, diag = Unnamed:7
rename_map = {
    "Unnamed: 1": "nda",
    "Unnamed: 2": "nom",
    "Unnamed: 3": "prenom",
    "Unnamed: 4": "sexe",
    "Unnamed: 5": "age",
    "Unnamed: 6": "date_adm",
    "Unnamed: 7": "diag"
}
df_concat.rename(columns=rename_map, inplace=True)

# --- Renommage des autres colonnes d'après vos indications ---
other_rename = {
    # Pour constituer l'anamnèse à la fois des colonnes "Histoire de la maladie" ou "anamnèse"
    # On ne renomme pas directement ici, la fusion se fera via regex.
    "Antécédents": "atcd_med",
    # Pour le traitement habituel (avant l'arrivée aux urgences)
    # On souhaite capter les colonnes avec "traitement habituel", "traitement à l'entrée", "traitement au domicile"
    # On ne renomme pas directement ici, la fusion se fera via regex.
    # Pour le traitement urgent, on veut "traitement administré" ou "actes thérapeutiques"
    # Pour l'évolution, la colonne contenant "evolution"
    # Pour la conclusion, "conclusion" (peut apparaître sous différentes formes)
    # Pour le score CCMU, "score CCMU"
    # Pour le diagnostic, on veut fusionner la colonne diag (déjà Unnamed:7) avec d'autres colonnes contenant "diagnostic"
    # Vous pouvez renommer d'autres colonnes si besoin.
}
df_concat.rename(columns=other_rename, inplace=True)


# --- Étape 3 : Fusion des colonnes via regex pour créer les variables calculées ---
def merge_columns_regex(df, regex_candidates, new_col_name):
    """
    Pour chaque ligne, recherche parmi les colonnes du DataFrame celles dont le nom
    correspond à l'un des motifs dans regex_candidates (re.IGNORECASE) et fusionne (combine_first)
    les valeurs non nulles dans une nouvelle colonne.
    """
    series = None
    for pattern in regex_candidates:
        matching_cols = [col for col in df.columns if re.search(pattern, col, re.IGNORECASE)]
        for col in matching_cols:
            if series is None:
                series = df[col]
            else:
                series = series.combine_first(df[col])
    df[new_col_name] = series
    return df

# Dictionnaire de mapping pour les variables calculées.
# Adaptez les regex selon vos intitulés exacts.
columns_mapping = {
    # anam_urg : à partir de "Histoire de la maladie" ou "anamnèse"
    "anam_urg": [r"histoire", r"anamn[eé]se"],
    # atcd_med : colonnes contenant "antécédents"
    "atcd_med": [r"antécédent\s"],
    # ttt_med : traitement habituel (avant arrivée) : 
    # on cherche explicitement "traitement habituel", "traitement à l'entrée" ou "traitement au domicile"
    "ttt_med": [r"traitement\s+(habituel|à l'entrée|au domicile)"],
    # ttt_urg : traitement urgences : "traitement administré" ou "actes thérapeutiques"
    "ttt_urg": [r"traitement\s+administr[ée]", r"actes\s+th[ée]rapeutiques","Traitement aux soins d'urgence"],
    # evol_urg : évolution aux urgences, on prend uniquement les colonnes contenant "evolution"
    "evol_urg": [r"[eé]volution"],
    # conclusion : conclusion du dossier (peut apparaître comme "conclusion" ou "conclusion pour urgences")
    "conclusion": [r"conclusion"],
    # ccmu : score CCMU
    "ccmu": [r"ccmu"],
    # diag : fusionner la colonne déjà renommée "diag" avec d'autres colonnes contenant "diagnostic"
    "diag": [r"diagnostic"]
}

for new_col, regex_list in columns_mapping.items():
    df_concat = merge_columns_regex(df_concat, regex_list, new_col)


# --- Étape 4 : Sélection des colonnes finales ---
# D'après votre liste finale attendue, nous conservons :
final_cols = [
    "nda",        # identifiant
    "nom",        # nom
    "prenom",     # prénom
    "sexe",       # sexe
    "age",        # âge
    "date_adm",   # date d'admission (provenant de Unnamed:6)
    "diag",       # diagnostic retenu (fusion de Unnamed:7 et autres colonnes "diagnostic")
    "anam_urg",   # anamnèse / histoire de la maladie
    "atcd_med",   # antécédents médicaux
    "ttt_med",    # traitement habituel (avant urgences)
    "ttt_urg",    # traitements administrés aux urgences (actes thérapeutiques)
    "evol_urg",   # évolution aux urgences
    "conclusion", # conclusion du dossier
    "ccmu"        # score CCMU
]
final_cols = [col for col in final_cols if col in df_concat.columns]
df_final = df_concat[final_cols].copy()

# --- Étape 5 : Réinitialisation de l'index ---
df_final = df_final.sort_values(by="date_adm")
df_final = df_final.dropna(how='all')
df_med = df_final.reset_index(drop=True)
df_med["nda"] = df_med["nda"].astype(str).str.replace(r'\.0$', '', regex=True)

print("\nAperçu du résultat final :")
print(df_med.head())
df_med.to_csv("df_med.csv")

In [ ]:
#### Radio ####

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import re

# --- Étape 1 : Lecture et concaténation des fichiers CGJ 073 ---
DATA_PATH = "/Bordeaux_CHU/"
pattern = os.path.join(DATA_PATH, "CGJ 073 -*")
files = glob.glob(pattern)

if not files:
    print("Aucun fichier correspondant trouvé.")
    exit()

print("Fichiers trouvés :")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
# Ici, on saute 3 lignes pour obtenir l'en‐tête souhaité
for file in files:
    try:
        df = pd.read_excel(file, skiprows=3)
        dfs.append(df)
    except Exception as e:
        print(f"Erreur lors de la lecture de {os.path.basename(file)} : {e}")

if not dfs:
    print("Aucun fichier n'a pu être traité.")
    exit()

df = pd.concat(dfs, ignore_index=True)
print(f"\nConcaténation terminée. Nombre total de lignes : {len(df)}")

# --- Étape 2 : Filtrage des doublons sur examen ---
# On identifie un examen par : Date heure examen, Libellé examen, Libellé Type examen
key_cols = ["Date heure examen", "Libellé examen", "Libellé Type examen"]

def filter_exams(group):
    cr = group["Compte rendu (cr) pour recherche texte (4000 caractères)"]
    has_cr = cr.notna() & (cr.astype(str).str.strip() != "")
    if has_cr.any():
        return group[has_cr]
    else:
        return group.iloc[[0]]

df_clean = df.groupby(key_cols, as_index=False).apply(filter_exams).reset_index(drop=True)

# --- Étape 3 : Préparation des colonnes d'identification ---
df_clean.rename(columns={
    "Numéro de venue (Xplore)": "nda",
    "Date Entree": "date_adm"
}, inplace=True)

# On ne conserve que les colonnes utiles pour le pivot
df_clean = df_clean[[
    "nda", "date_adm", "Numéro examen", "Date heure examen",
    "Libellé examen", "Type examen",
    "Compte rendu (cr) pour recherche texte (4000 caractères)"
]]

# --- Étape 4 : Création d'un index d'examen par patient et par type d'examen ---
df_clean["exam_index"] = df_clean.groupby(["nda", "Type examen"]).cumcount() + 1

# --- Étape 5 : Construction des nouveaux noms de colonnes dynamiques ---
# Pour chaque examen, on crée :
# - Une colonne pour le numéro d'examen
# - Une colonne pour le libellé de l'examen
# - Une colonne pour la date de l'examen
# - Une colonne pour le compte rendu
df_clean["new_num_col"] = df_clean.apply(lambda r: f"{r['Type examen']}_num{r['exam_index']}", axis=1)
df_clean["new_examen_col"] = df_clean.apply(lambda r: f"{r['Type examen']}_examen{r['exam_index']}", axis=1)
df_clean["new_date_col"] = df_clean.apply(lambda r: f"{r['Type examen']}_date{r['exam_index']}", axis=1)
df_clean["new_cr_col"] = df_clean.apply(lambda r: f"{r['Type examen']}_cr{r['exam_index']}", axis=1)

# --- Étape 6 : Pivot des données en format large ---
df_num = df_clean.pivot(index=["nda", "date_adm"], columns="new_num_col", values="Numéro examen")
df_examen = df_clean.pivot(index=["nda", "date_adm"], columns="new_examen_col", values="Libellé examen")
df_date = df_clean.pivot(index=["nda", "date_adm"], columns="new_date_col", values="Date heure examen")
df_cr = df_clean.pivot(index=["nda", "date_adm"], columns="new_cr_col", values="Compte rendu (cr) pour recherche texte (4000 caractères)")

df_wide = df_date.join([df_num, df_examen, df_cr])
df_wide.reset_index(inplace=True)

# --- Étape 7 : Réorganisation des colonnes ---
# On souhaite que, pour chaque groupe d'examen (pour chaque Type examen et exam_index),
# les colonnes apparaissent dans l'ordre : date, numéro, libellé, compte rendu.
# Nous allons extraire ces colonnes et les trier avec une fonction de clé personnalisée.

def sort_key(col):
    # On attend des noms du type : {Type examen}_{field}{index}
    # field attendu : "date", "num", "examen", "cr"
    m = re.match(r"^(?P<type>.+)_(?P<field>date|num|examen|cr)(?P<index>\d+)$", col)
    if m:
        exam_type = m.group("type").lower()
        field = m.group("field")
        index = int(m.group("index"))
        field_order = {"date": 0, "num": 1, "examen": 2, "cr": 3}
        return (exam_type, index, field_order[field])
    else:
        return (col,)

# Identifier les colonnes d'examen (hors nda et date_adm)
non_id_cols = [col for col in df_wide.columns if col not in ["nda", "date_adm"]]
exam_cols_sorted = sorted(non_id_cols, key=sort_key)

# Réassembler l'ordre final
final_order = ["nda", "date_adm"] + exam_cols_sorted
df_wide = df_wide[final_order]

# --- Étape 8 : Suppression des lignes totalement vides (hors nda et date_adm) ---
df_wide = df_wide.dropna(how="all", subset=[c for c in df_wide.columns if c not in ["nda", "date_adm"]])

# --- Optionnel : Réinitialisation de l'index final ---
df_radio = df_wide.reset_index(drop=True)
df_radio["nda"] = df_radio["nda"].astype(str).str.replace(r'\.0$', '', regex=True)

print("\nAperçu du tableau final :")
print(df_radio.head())
df_radio.to_csv("df_radio.csv")

In [ ]:
df_radio

In [ ]:
#### Biologie ####

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

#######################################################
# A) APPEND TOUS LES FICHIERS SYNERGY (brut)
#######################################################
def append_all_synergy():
    synergy_path = "/Bio/synergy/"
    synergy_files = glob.glob(os.path.join(synergy_path, "*.xlsx"))

    if not synergy_files:
        print("[Synergy] Aucun fichier trouvé.")
        return pd.DataFrame()

    dfs = []
    for f in synergy_files:
        try:
            # Lecture brut, skiprows=1 si nécessaire
            df = pd.read_excel(f, sheet_name="Nombre", skiprows=1)
            dfs.append(df)
        except Exception as e:
            print(f"[Synergy] Erreur lecture {f} : {e}")

    if not dfs:
        print("[Synergy] Aucun DF concaténé.")
        return pd.DataFrame()

    df_syn_raw = pd.concat(dfs, ignore_index=True)
    print(f"[Synergy raw] => {df_syn_raw.shape[0]} lignes, {df_syn_raw.shape[1]} colonnes.")
    return df_syn_raw

#######################################################
# B) TRAITEMENT SYNERGY
#######################################################
def process_synergy(df_syn_raw):
    """
    On suppose que Synergy est déjà au format large :
    - On renomme les colonnes
    - On conserve seulement les colonnes cibles
    - On convertit les dates
    """
    # Dictionnaire de renommage Synergy
    SYNERGY_RENAME = {
        "Unnamed: 1": "nda",
        "Unnamed: 2": "date",
        "Unnamed: 3": "date_prlvt",
        "N° venue": "nda",
        "Date Prélèvement (en date)": "date_bio",

        "C1CRS CREATININE sg": "creat",
        "C1KS POTASSIUM sg": "potassium",
        "C1NAS SODIUM sg": "sodium",
        "C1CAB CALCIUM sg arsenazo": "calcium",
        "C3TNI TROPONINE I": "tropo",
        "C2CRP CRP": "crp",
        "C2TGO ASAT(TGO)": "asat",
        "C2TGP ALAT(TGP)": "alat",
        "C1BTS BILIRUBINE totale": "bili_tot",
        "C2LIS LIPASE sg": "lipase",
        "HGB Leucocytes": "leuco",
        "HFSNV PN valeur absolue": "neutro",
        "HFSMV Mono valeur absolue": "monocytes",
        "HFSLV Lymp valeur absolue": "lympho",
        "HFSB  Basophiles": "basophiles",
        "HFSE  Eosinophiles": "eosinophiles",
        "HHB  Hémoglobine": "hemoglobine",
        "HVGM  VGM": "vgm",
        "HPLAQ  Plaquettes": "plaq",
        "HTPS Taux Prothrombine": "TP",
        "HINR INR": "inr",
        "HDDEP  DD ELISA/ T": "ddimere",
    }

    # Liste de colonnes cibles
    SYNERGY_COLS = [
        "nda", "date", "date_prlvt", "date_bio",
        "creat", "potassium", "sodium", "calcium",
        "tropo", "crp", "asat", "alat", "bili_tot",
        "lipase", "leuco", "neutro", "monocytes",
        "lympho", "basophiles", "eosinophiles",
        "hemoglobine", "vgm", "plaq", "TP", "inr",
        "ddimere"
    ]

    # Renommage
    df_syn_raw.rename(columns=SYNERGY_RENAME, inplace=True, errors="ignore")

    # Conversion nda en string
    if "nda" in df_syn_raw.columns:
        df_syn_raw["nda"] = df_syn_raw["nda"].astype(str)

    # Convertir date_prlvt / date_bio / date
    for date_col in ["date_prlvt", "date_bio", "date"]:
        if date_col in df_syn_raw.columns:
            df_syn_raw[date_col] = pd.to_datetime(df_syn_raw[date_col], errors="coerce", dayfirst=True)

    # Filtrer
    keep = [c for c in SYNERGY_COLS if c in df_syn_raw.columns]
    df_syn = df_syn_raw[keep].copy()

    print(f"[Synergy processed] => {df_syn.shape[0]} lignes, {df_syn.shape[1]} colonnes.")
    return df_syn

#######################################################
# C) APPEND TOUS LES FICHIERS GLIMS (brut)
#######################################################
def append_all_glims():
    glims_path = "/Bio/glims/"
    glims_files = glob.glob(os.path.join(glims_path, "*.xlsx"))

    if not glims_files:
        print("[Glims] Aucun fichier trouvé.")
        return pd.DataFrame()

    dfs = []
    for f in glims_files:
        try:
            # Lecture brut, skiprows=1
            df = pd.read_excel(f, sheet_name="Analyses", skiprows=1)
            dfs.append(df)
        except Exception as e:
            print(f"[Glims] Erreur lecture {f} : {e}")

    if not dfs:
        print("[Glims] Aucun DF concaténé.")
        return pd.DataFrame()

    df_glims_raw = pd.concat(dfs, ignore_index=True)
    print(f"[Glims raw] => {df_glims_raw.shape[0]} lignes, {df_glims_raw.shape[1]} colonnes.")
    return df_glims_raw

#######################################################
# D) TRAITEMENT GLIMS (pivot)
#######################################################
def process_glims_pivot(df_glims_raw):
    """
    On a un format 4 colonnes : nda, date_prlvt, analyse, resultat
    On pivot => index=(nda, date_prlvt), columns=analyse, values=resultat
    On renomme => creat, potassium, etc.
    """

    # Renommage colonnes de base
    rename_base = {
        "N° venue": "nda",
        "Date Prélèvement (en date)": "date_prlvt",
        "Libellé analyse détaillée": "analyse",
        "Résultat brute de l'analyse": "resultat",
    }
    df_glims_raw.rename(columns=rename_base, inplace=True, errors="ignore")

    if "nda" in df_glims_raw.columns:
        df_glims_raw["nda"] = df_glims_raw["nda"].astype(str)
    if "date_prlvt" in df_glims_raw.columns:
        df_glims_raw["date_prlvt"] = pd.to_datetime(df_glims_raw["date_prlvt"], errors="coerce", dayfirst=True)

    # On ne garde que 4 colonnes
    base_cols = ["nda", "date_prlvt", "analyse", "resultat"]
    keep = [c for c in base_cols if c in df_glims_raw.columns]
    df_g = df_glims_raw[keep].copy()

    # Map analyses => noms finaux
    analyses_map = {
        "Créatinine sg": "creat",
        "Potassium sg": "potassium",
        "Sodium sg": "sodium",
        "Calcium sg arsenazo": "calcium",
        "Troponine I HS": "tropo",
        "CRP": "crp",
        "ASAT (TGO)": "asat",
        "ALAT (TGP)": "alat",
        "Bilirubine totale": "bili_tot",
        "Lipase sg": "lipase",
        "Leucocytes": "leuco",
        "PNeutro Va": "neutro",
        "Lympho Va": "lympho",
        "Monocytes Va": "monocytes",
        "PBaso Va": "basophiles",
        "PEosino Va": "eosinophiles",
        "Hémoglobine": "hemoglobine",
        "PlaquettesEDTA": "plaq",
        "TP Taux Prothrombine": "TP",
        "Activité AXa (HNF)": "axa_hnf",
        "Activité AXa Xarelto": "axa_xarelto",
        "Activité AXa Eliquis": "axa_eliquis",
        "Activité AXa HBPM": "axa_hbpm",
        "Anti-IIa Pradaxa": "aiia_pradaxa",
        "Activité AXa Arixtra": "axa_arixtra",
        "Activité AXa Orgaran": "axa_orgaran",
        "D-Dimères dosage": "ddimere",
    }

    # Filtrer sur analyses
    allowed_analyses = set(analyses_map.keys())
    df_g = df_g[df_g["analyse"].isin(allowed_analyses)].copy()

    # Pivot
    df_pivot = df_g.pivot_table(
        index=["nda", "date_prlvt"],
        columns="analyse",
        values="resultat",
        aggfunc="first"
    ).reset_index()

    # Renommer colonnes pivotées
    rename_pivot = {}
    for orig, newcol in analyses_map.items():
        if orig in df_pivot.columns:
            rename_pivot[orig] = newcol
    df_pivot.rename(columns=rename_pivot, inplace=True, errors="ignore")

    print(f"[Glims pivot final] => {df_pivot.shape[0]} lignes, {df_pivot.shape[1]} colonnes.")
    return df_pivot

#######################################################
# E) Fonction de remplissage => 1 date par nda
#######################################################
def fill_from_subsequent_rows(group):
    """
    group = data pour un nda, trié par date_prlvt asc.
    On part de la première date, et on remplit les colonnes NaN
    avec les valeurs trouvées dans les dates ultérieures.
    """
    if len(group) == 1:
        return group.iloc[0]

    filled = group.iloc[0].copy()
    for i in range(1, len(group)):
        row = group.iloc[i]
        for col in group.columns:
            if col == "date_prlvt":
                continue  # on garde la plus ancienne date
            if pd.isna(filled[col]) and not pd.isna(row[col]):
                filled[col] = row[col]
    return filled

#######################################################
# MAIN
#######################################################
def main():
    # 1) Synergy
    print("=== Lecture Synergy (append brut) ===")
    df_syn_raw = append_all_synergy()
    print("\n=== Traitement Synergy ===")
    df_syn = process_synergy(df_syn_raw)

    # 2) Glims
    print("\n=== Lecture Glims (append brut) ===")
    df_glims_raw = append_all_glims()
    print("\n=== Traitement Glims (pivot) ===")
    df_glims = process_glims_pivot(df_glims_raw)

    # 3) Append vertical
    print("\n=== Concat Synergy + Glims ===")
    df_bio = pd.concat([df_syn, df_glims], ignore_index=True)
    print(f"[Append synergy+glims] => {df_bio.shape[0]} lignes, {df_bio.shape[1]} colonnes.")

    # 4) Tri par nda, date_prlvt
    if "date_prlvt" in df_bio.columns:
        df_bio["date_prlvt"] = pd.to_datetime(df_bio["date_prlvt"], errors="coerce")
        df_bio.sort_values(by=["nda", "date_prlvt"], inplace=True)

    # 5) fill => 1 date par nda, on remplit
    print("\n=== Remplissage => 1 date par nda ===")
    df_bio_filled = df_bio.groupby("nda", as_index=False).apply(fill_from_subsequent_rows)
    df_bio_filled.reset_index(drop=True, inplace=True)

    print(f"[df_bio_filled] => {df_bio_filled.shape[0]} lignes, {df_bio_filled.shape[1]} colonnes.")
    print(df_bio_filled.head(20))
    return df_bio_filled

if __name__ == "__main__":
    df_final = main()
    print("\n--- Script terminé. ---")

In [ ]:
df_bio.to_csv("df_bio.csv")

In [ ]:
#### données administratives ####

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

######################################
# Partie 1 : Lecture et traitement des fichiers CGJ 084
######################################
path_084 = "/Bordeaux_CHU/"
pattern_084 = os.path.join(path_084, "CGJ 084 -*.xlsx")
files_084 = glob.glob(pattern_084)
dfs_084 = []
for f in files_084:
    try:
        # skiprows=1 pour CGJ 084
        df = pd.read_excel(f, skiprows=1)
        dfs_084.append(df)
    except Exception as e:
        print(f"Erreur lecture {f} : {e}")

if dfs_084:
    df_084 = pd.concat(dfs_084, ignore_index=True)
    rename_map_084 = {
        "UG entrée séjour Code": "uam_service",
        "Nda": "nda",
        "Date entrée UG entrée séjour": "date_entree",
        "Nom du patient": "nom",
        "Prénom du patient": "prenom",
        "Date de naissance du patient": "date_naissance",
        "Libelle de la ville de naissance": "ville_naissance",
        "Pays Naissance": "pays_naissance",
        "Numéro de securite sociale": "nir",
        "Adresse du patient (rue)": "adresse_rue",
        "Code postal de la ville": "adresse_cp",
        "Code commune de la ville": "adresse_insee",
        "Libelle de la ville": "adresse_ville",
    }
    df_084.rename(columns=rename_map_084, inplace=True, errors="ignore")

    cols_to_drop_084 = [
        "Date et heure de création",
        "Date et heure de modification",
        "Date de dernière modification",
        "Libellé du Hameau/Lieu-dit",
        "Identifiant de la categorie professionnelle",
        "Statut INS",
        "Code statut INS",
        "Libellé Statut INS"
    ]
    df_084.drop(columns=[c for c in cols_to_drop_084 if c in df_084.columns],
                inplace=True, errors="ignore")

else:
    df_084 = pd.DataFrame()
    print("Aucun fichier CGJ 084 trouvé ou lisible.")


######################################
# Partie 2 : Lecture et traitement des fichiers CGJ 085
######################################
path_085 = "/Bordeaux_CHU/"
pattern_085 = os.path.join(path_085, "CGJ 085 -*.xlsx")
files_085 = glob.glob(pattern_085)
dfs_085 = []
for f in files_085:
    try:
        # skiprows=3 pour CGJ 085
        df = pd.read_excel(f, skiprows=3)
        dfs_085.append(df)
    except Exception as e:
        print(f"Erreur lecture {f} : {e}")

if dfs_085:
    df_085 = pd.concat(dfs_085, ignore_index=True)
    rename_map_085 = {
        "UAM entrée venue": "uam_service",
        "No venue": "nda",
        "Nom usuel patient": "nom",
        "Date de naissance": "date_naissance",
        "Ville de naissance": "ville_naissance",
        "Pays de naissance": "pays_naissance",
        "Prénom usuel patient": "prenom",
        "Date d'entrée venue": "date_entree",
        "Numéro SS assuré": "nir",
        "1ere ligne adresse": "adresse_rue",
        "Code postal": "adresse_cp",
        "Ville de résidence": "adresse_ville",
    }
    df_085.rename(columns=rename_map_085, inplace=True, errors="ignore")

    cols_to_drop_085 = [
        "Date création dossier",
        "Date modification dossier",
        "2ieme ligne adresse",
        "Catégorie socio-professionnelle"
    ]
    df_085.drop(columns=[c for c in cols_to_drop_085 if c in df_085.columns],
                inplace=True, errors="ignore")

else:
    df_085 = pd.DataFrame()
    print("Aucun fichier CGJ 085 trouvé ou lisible.")


######################################
# Partie 3 : Concaténation et suppression des doublons
######################################
df_final = pd.concat([df_084, df_085], ignore_index=True)

if df_final.empty:
    print("Aucun enregistrement final.")
else:
    # Supprimer les doublons sur 'nda' en conservant la ligne avec le plus de valeurs non-null
    df_final["count_nonnull"] = df_final.notna().sum(axis=1)
    df_final.sort_values(by="count_nonnull", ascending=False, inplace=True)
    df_final.drop_duplicates(subset="nda", keep="first", inplace=True)
    df_final.drop(columns=["count_nonnull"], inplace=True, errors="ignore")

    # Tri par date_entree si la colonne existe
    if "date_entree" in df_final.columns:
        df_final.sort_values(by="date_entree", ascending=True, inplace=True)

    # Renommer le DataFrame final en df_admin
    df_admin = df_final.reset_index(drop=True)

    print("\nAperçu du df_admin :")
    print(df_admin.head())
    df_admin.to_csv("df_admin.csv")

In [ ]:
#### questionnaires imagerie ####